In [ ]:
import os  
import sys  
import shutil  
import tempfile  
import subprocess  
from pathlib import Path  
from typing import List, Dict  

In [ ]:

import resource
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY)) 

In [ ]:
import os  

def rename_fch_to_fchk(directory: str = ".") -> None:
    
    
    directory = os.path.abspath(directory)  
    print(f"Checking directory: {directory}")  

    
    for name in os.listdir(directory):  
        old_path = os.path.join(directory, name)  

        
        if not os.path.isfile(old_path):  
            continue  

        
        if name.endswith(".fch"):  
            
            new_name = name[:-4] + ".fchk"  
            new_path = os.path.join(directory, new_name)  

            
            if os.path.exists(new_path):  
                print(f"Skip: {old_path} -> {new_path} (target already exists)")  
                continue  

            
            os.rename(old_path, new_path)  
            print(f"Renamed: {old_path} -> {new_path}")  

    print("Renaming finished.")  


rename_fch_to_fchk(".")

In [ ]:
def _run_subprocess(cmd: List[str],
                    input_text: str = None,
                    cwd: Path = None,
                    timeout: int = None) -> subprocess.CompletedProcess:
    
    
    stdin_bytes = input_text.encode('utf-8') if isinstance(input_text, str) else None  
    try:
        
        proc = subprocess.run(  
            cmd,  
            input=stdin_bytes,  
            cwd=str(cwd) if cwd else None,  
            capture_output=True,  
            timeout=timeout,  
            check=True  
        )
        return proc  
    except subprocess.CalledProcessError as e:
        
        raise RuntimeError(
            "Public message removed for release."
            "Public message removed for release."
            f"STDOUT：\n{e.stdout.decode(errors='ignore')}\n"
            f"STDERR：\n{e.stderr.decode(errors='ignore')}\n"
        ) from e
    except subprocess.TimeoutExpired as e:
        
        raise TimeoutError("Public message removed for release.") from e


def _ensure_executable(cmd_hint: str) -> str:
    
    
    p = Path(cmd_hint)
    if p.exists():
        return str(p.resolve())  

    
    which = shutil.which(cmd_hint)  
    if which:
        return which  

    
    raise FileNotFoundError("Public message removed for release.")


def vmd_render_with_showorb(
    cub_homo: Path,
    cub_lumo: Path,
    showorb_vmd: Path,
    vmd_cmd: str = "vmd",
    timeout_vmd: int = 300
) -> Dict[str, Path]:
    
    
    cub_homo = cub_homo.resolve()  
    cub_lumo = cub_lumo.resolve()  
    showorb_vmd = showorb_vmd.resolve()  

    
    if not cub_homo.exists():
        raise FileNotFoundError("Public message removed for release.")  
    if not cub_lumo.exists():
        raise FileNotFoundError("Public message removed for release.")  
    if not showorb_vmd.exists():
        raise FileNotFoundError("Public message removed for release.")  

    
    homo_tga = cub_homo.with_suffix(".tga")  
    lumo_tga = cub_lumo.with_suffix(".tga")  
    homo_vmd = cub_homo.with_suffix(".vmd")  
    lumo_vmd = cub_lumo.with_suffix(".vmd")  

    
    tcl_script = .strip()  

    
    with tempfile.NamedTemporaryFile(suffix=".tcl", prefix="vmd_batch_", delete=False, mode="w", encoding="utf-8") as tf:
        tf.write(tcl_script)  
        tcl_path = Path(tf.name)  

    try:
        
        vmd_exec = _ensure_executable(vmd_cmd)  
        
        cmd = [vmd_exec, "-dispdev", "text", "-e", str(tcl_path), "-eofexit"]  
        _run_subprocess(cmd, cwd=cub_homo.parent, timeout=timeout_vmd)  
    finally:
        
        try:
            tcl_path.unlink(missing_ok=True)  
        except Exception:
            pass  

    
    return {
        "homo_tga": homo_tga,  
        "lumo_tga": lumo_tga,  
        "homo_vmd": homo_vmd,  
        "lumo_vmd": lumo_vmd   
    }


def process_checkpoints(
    check_point_file_path: str,
    multiwfn_cmd: str = "Multiwfn",
    vmd_cmd: str = "vmd",
    showorb_vmd: str = "showorb.vmd",
    timeout_multiwfn: int = 600,
    timeout_vmd: int = 300
) -> List[Dict[str, Path]]:
    
    
    root = Path(check_point_file_path).expanduser().resolve()  
    if not root.exists():
        raise FileNotFoundError("Public message removed for release.")  

    
    multiwfn_exec = _ensure_executable(multiwfn_cmd)  
    _ = _ensure_executable(vmd_cmd)  

    
    chk_files = [  
        p for p in root.rglob("*")  
        if p.is_file() and p.suffix.lower() in (".fchk", ".molden")  
    ]
    if not chk_files:
        raise FileNotFoundError("Public message removed for release.")  

    results: List[Dict[str, Path]] = []  

    
    for chk in chk_files:
        
        workdir = chk.parent  
        filename = chk.stem  
        print("Public message removed for release.")  

        
        
        homo_cmds = [  
            str(chk),  
            "200",     
            "3",       
            "h",       
            "2",       
            "0",       
            "q"        
        ]
        homo_input = "\n".join(homo_cmds) + "\n"  
        _run_subprocess([multiwfn_exec], input_text=homo_input, cwd=workdir, timeout=timeout_multiwfn)  

        
        lumo_cmds = [  
            str(chk),  
            "200",     
            "3",       
            "l",       
            "2",       
            "0",       
            "q"        
        ]
        lumo_input = "\n".join(lumo_cmds) + "\n"  
        _run_subprocess([multiwfn_exec], input_text=lumo_input, cwd=workdir, timeout=timeout_multiwfn)  

        
        
        src_h = workdir / "h.cub"  
        src_l = workdir / "l.cub"  
        if not src_h.exists():
            raise FileNotFoundError("Public message removed for release.")
        if not src_l.exists():
            raise FileNotFoundError("Public message removed for release.")

        dst_h = workdir / f"{filename}_HOMO.cub"  
        dst_l = workdir / f"{filename}_LUMO.cub"  
        if dst_h.exists():
            dst_h.unlink()  
        if dst_l.exists():
            dst_l.unlink()  
        shutil.move(str(src_h), str(dst_h))  
        shutil.move(str(src_l), str(dst_l))  

        
        out = vmd_render_with_showorb(  
            cub_homo=dst_h,  
            cub_lumo=dst_l,  
            showorb_vmd=Path(showorb_vmd),  
            vmd_cmd=vmd_cmd,  
            timeout_vmd=timeout_vmd  
        )

        
        results.append({  
            "checkpoint": chk,            
            "filename": filename,         
            "homo_cub": dst_h,            
            "lumo_cub": dst_l,            
            "homo_tga": out["homo_tga"],  
            "lumo_tga": out["lumo_tga"],  
            "homo_vmd": out["homo_vmd"],  
            "lumo_vmd": out["lumo_vmd"],  
        })

    
    return results

In [ ]:




from pathlib import Path

results = process_checkpoints(
    check_point_file_path=".",
    multiwfn_cmd="Multiwfn",                 
    vmd_cmd="vmd",                           
    showorb_vmd="showorb.vmd",      
    timeout_multiwfn=60000,
    timeout_vmd=30000
)